# CNN Baseline — QSO

Usa `PaddedSpectralCNN` de [`src/models/cnn.py`](../../src/models/cnn.py).


## 1. Setup

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, print_info

OBJECT_TYPE = "QSO"
print_info()

paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths["spectra_h5"].with_name(f"{OBJECT_TYPE}spectra_padded.h5")
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"HDF5:   {HDF5_PATH}")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split, apply_split
from src.models import PaddedSpectralCNN
from src.evaluation import compute_redshift_metrics, metrics_by_redshift_bin
from config import SPLITS_DIR

RESULTS_DIR = PROJECT_RESULTS / OBJECT_TYPE / "cnn_baseline"
MODEL_DIR   = MODELS_DIR / OBJECT_TYPE / "cnn_baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"MODEL_DIR:   {MODEL_DIR}")

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs visiveis: {len(gpus)}  ({[g.name for g in gpus]})")


## 2. Dados

In [ ]:
X, y, n_wave = load_spectral_dataset(HDF5_PATH)
X = normalize_spectra(X)
print(f"X: {X.shape}  y: {y.shape}  n_wave: {n_wave}")

# Split centralizado ESTRATIFICADO por z. Salva indices em splits/{OBJ}_split.npz
# para reuso entre todos os notebooks (mesmo test set em todos os modelos).
train_idx, val_idx, test_idx = make_or_load_split(OBJECT_TYPE, y, splits_dir=SPLITS_DIR)
X_train, X_val, X_test = apply_split(X, train_idx, val_idx, test_idx)
y_train, y_val, y_test = apply_split(y, train_idx, val_idx, test_idx)
print(f"Train: {len(y_train):,}  Val: {len(y_val):,}  Test: {len(y_test):,}")

## 3. Treinar

In [ ]:
cnn = PaddedSpectralCNN(n_wave=n_wave, learning_rate=3e-4)
cnn.build()
cnn.model.summary()

cnn.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,   # val explicito (nao usa validation_split do Keras)
    epochs=50, batch_size=64,
    patience_es=20, patience_lr=10,
    verbose=1,
)

y_pred_test = cnn.predict(X_test)
results = compute_redshift_metrics(y_test, y_pred_test)
print(f"\n{'='*50}")
print(f"Resultados em TEST  (n={len(y_test):,}, train_time={cnn.train_time:.0f}s)")
print(f"{'='*50}")
print(f"  RMSE       : {results['rmse']:.4f}")
print(f"  MAE        : {results['mae']:.4f}")
print(f"  R2         : {results['r2']:.4f}")
print(f"  bias       : {results['bias']:+.4e}")
print(f"  sigma_NMAD : {results['nmad']:.4e}")
print(f"  outliers (|Δz/(1+z)|>0.05) : {results['outliers_0.05_pct']:.2f}%")
print(f"  outliers (|Δz/(1+z)|>0.15) : {results['outliers_0.15_pct']:.2f}%")

## 4. Plots — curvas de treino e predicao

In [ ]:
history = cnn.history.history
epochs = np.arange(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"CNN baseline — {OBJECT_TYPE}", fontsize=13, fontweight="bold")

# Loss
axes[0].plot(epochs, history["loss"],     label="Train", lw=2)
axes[0].plot(epochs, history["val_loss"], label="Val",   lw=2)
axes[0].set_yscale("log")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE")
axes[0].set_title("Loss"); axes[0].legend()

# RMSE
axes[1].plot(epochs, history["rmse"],     label="Train", lw=2)
axes[1].plot(epochs, history["val_rmse"], label="Val",   lw=2)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("RMSE")
axes[1].set_title("RMSE"); axes[1].legend()

# Predito vs Real
y_pred = y_pred_test
sc = axes[2].scatter(y_test, y_pred, c=y_test, cmap="viridis", alpha=0.5, s=8)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=1.5, label="Ideal")
axes[2].set_xlabel("z_true"); axes[2].set_ylabel("z_pred")
axes[2].set_title(f"R²={results['r2']:.4f}  σ_NMAD={results['nmad']:.4f}")
axes[2].legend()
plt.colorbar(sc, ax=axes[2], label="z_true")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_and_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta_z = results["delta_z"]
bins = metrics_by_redshift_bin(y_test, y_pred_test, n_bins=10)["bins"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Residuos — {OBJECT_TYPE}", fontsize=13, fontweight="bold")

# Histograma de delta_z
axes[0].hist(delta_z, bins=80, color="steelblue", alpha=0.8)
for thr, c in [(0.05, "orange"), (0.15, "red")]:
    axes[0].axvline( thr, ls="--", c=c, label=f"|Δz/(1+z)|={thr}")
    axes[0].axvline(-thr, ls="--", c=c)
axes[0].axvline(results["bias"], color="k", lw=1.5, label=f"bias={results['bias']:+.2e}")
axes[0].set_xlabel("Δz / (1+z)")
axes[0].set_title("Distribuicao dos residuos")
axes[0].legend()

# σ_NMAD por bin de z
zc = [b["z_center"] for b in bins]
nmad = [b["nmad"]   for b in bins]
axes[1].bar(zc, nmad, width=(zc[1]-zc[0])*0.85, color="steelblue", alpha=0.8)
axes[1].axhline(results["nmad"], ls="--", color="k", label=f"global={results['nmad']:.2e}")
axes[1].set_xlabel("z_true (centro do bin)")
axes[1].set_ylabel("σ_NMAD")
axes[1].set_title("σ_NMAD por faixa de z")
axes[1].legend()

# % outliers (|Δz/(1+z)|>0.15) por bin de z
out = [b["outliers_pct"] for b in bins]
axes[2].bar(zc, out, width=(zc[1]-zc[0])*0.85, color="salmon", alpha=0.8)
axes[2].axhline(results["outliers_pct"], ls="--", color="k",
                label=f"global={results['outliers_pct']:.2f}%")
axes[2].set_xlabel("z_true (centro do bin)")
axes[2].set_ylabel("Outliers (%)")
axes[2].set_title("% |Δz/(1+z)| > 0.15 por faixa de z")
axes[2].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "residuals_by_z.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame(bins).round(5)

## 5. Salvar modelo + predicoes + metricas

In [ ]:
cnn.save(MODEL_DIR / "cnn_baseline.keras")

np.savez(
    RESULTS_DIR / "predictions.npz",
    y_test=y_test, y_pred=y_pred_test,
    delta_z=results["delta_z"], test_idx=test_idx,
)

scalar_keys = ["rmse", "mae", "r2", "bias", "nmad",
               "outliers_pct", "outliers_0.05_pct", "outliers_0.15_pct"]
row = {"model": "cnn_baseline", "object": OBJECT_TYPE,
       "n_test": len(y_test), "train_time_s": cnn.train_time,
       **{k: results[k] for k in scalar_keys}}
pd.DataFrame([row]).to_csv(RESULTS_DIR / "metrics.csv", index=False)
print(f"Salvo:\n  {MODEL_DIR / 'cnn_baseline.keras'}\n  {RESULTS_DIR / 'predictions.npz'}\n  {RESULTS_DIR / 'metrics.csv'}")